# NetCDF Extraction to CSV (Bremen)

## Main objective
Extract local daily precipitation from a large E-OBS NetCDF file, save it as CSV, and perform basic pandas quality checks.


### Important interpretation note
E-OBS provides **daily precipitation totals** for grid cells. This supports resilience screening and educational analysis, not sub-hourly hydraulic sewer design.

In [1]:
# Import the main libraries used in this notebook.
# pathlib handles file paths safely on Windows.
# xarray reads NetCDF data.
# pandas is used for tabular checks and CSV export.
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr


In [ ]:
# Define project paths.
# You can change these paths if your folder location is different.
project_root = Path(r"C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\vro_project")
nc_path = project_root / "data" / "raw" / "rr_ens_mean_0.1deg_reg_v33.0e.nc"
output_csv = project_root / "data" /"processed" / "bremen_daily_precipitation.csv"

# Study location: Bremen, Germany.
# These are the target coordinates used to find the nearest E-OBS grid point.
target_lat = 53.071895
target_lon = 8.794528

# Define a small regional box around the city.
# We will average all selected grid cells in this box to get a city-scale daily series.
bbox = {
    "min_lat": target_lat - 0.2,
    "max_lat": target_lat + 0.2,
    "min_lon": target_lon - 0.25,
    "max_lon": target_lon + 0.25,
}

print("NetCDF file exists:", nc_path.exists())
print("Output CSV path:", output_csv)


NetCDF file exists: True
Output CSV path: C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project\data\processed\bremen_daily_precipitation.csv


In [3]:
# Open the NetCDF dataset.
# This does not yet convert the whole dataset to pandas.
# First inspect structure so we do not make wrong assumptions.
ds = xr.open_dataset(nc_path)

print("Data variables:", list(ds.data_vars))
print("Dimensions:", dict(ds.sizes))
print("Coordinates:", list(ds.coords))
print("Global attribute keys (first 10):", list(ds.attrs.keys())[:10])


Data variables: ['rr']
Dimensions: {'time': 27759, 'latitude': 465, 'longitude': 705}
Coordinates: ['time', 'longitude', 'latitude']
Global attribute keys (first 10): ['CDI', 'CDO', 'E-OBS_version', 'Conventions', 'References', 'history', 'NCO']


In [4]:
# Detect precipitation variable safely.
# If there is only one data variable, use it.
# If there are many variables, search metadata for words like rain or precip.
if len(ds.data_vars) == 1:
    precip_var = list(ds.data_vars)[0]
else:
    candidates = []
    for name in ds.data_vars:
        attrs_text = " ".join(str(v).lower() for v in ds[name].attrs.values())
        if ("rain" in attrs_text) or ("precip" in attrs_text):
            candidates.append(name)
    precip_var = candidates[0] if candidates else list(ds.data_vars)[0]

print("Selected precipitation variable:", precip_var)
print("Variable attributes:")
for k, v in ds[precip_var].attrs.items():
    print(f"  {k}: {v}")


Selected precipitation variable: rr
Variable attributes:
  standard_name: thickness_of_rainfall_amount
  long_name: rainfall
  units: mm


In [5]:
# Detect coordinate names for time, latitude, and longitude.
# This keeps the notebook robust if datasets use slightly different naming.
time_name = next(c for c in ds.coords if "time" in c.lower())
lat_name = next(c for c in ds.coords if c.lower() in ("lat", "latitude", "y"))
lon_name = next(c for c in ds.coords if c.lower() in ("lon", "longitude", "x"))

print("Detected coordinate names:")
print("  time ->", time_name)
print("  latitude ->", lat_name)
print("  longitude ->", lon_name)


Detected coordinate names:
  time -> time
  latitude -> latitude
  longitude -> longitude


## Compact inspection block (table style views)

Use this section to see what NetCDF data looks like before extraction.
These checks convert only small slices to pandas tables, not the whole dataset.

In [6]:
# Quick structural view.
# This prints a compact xarray summary of dimensions and variable layout.
print(ds)
print()
print(ds[precip_var])

<xarray.Dataset> Size: 36GB
Dimensions:    (time: 27759, latitude: 465, longitude: 705)
Coordinates:
  * time       (time) datetime64[ns] 222kB 1950-01-01 1950-01-02 ... 2025-12-31
  * latitude   (latitude) float64 4kB 25.05 25.15 25.25 ... 71.25 71.35 71.45
  * longitude  (longitude) float64 6kB -24.95 -24.85 -24.75 ... 45.35 45.45
Data variables:
    rr         (time, latitude, longitude) float32 36GB ...
Attributes:
    CDI:            Climate Data Interface version 2.5.3 (https://mpimet.mpg....
    CDO:            Climate Data Operators version 2.5.3 (https://mpimet.mpg....
    E-OBS_version:  33.0e
    Conventions:    CF-1.4
    References:     http://surfobs.climate.copernicus.eu/dataaccess/access_eo...
    history:        Tue Mar 10 16:37:38 2026: ncks -O --no-abc -d time,0,2775...
    NCO:            netCDF Operators version 5.3.3 (Homepage = http://nco.sf....

<xarray.DataArray 'rr' (time: 27759, latitude: 465, longitude: 705)> Size: 36GB
[9100094175 values with dtype=float32]

In [7]:
# Table-like view 1: one day across many grid cells.
# This helps you see latitude, longitude, and precipitation columns.
one_day = ds[precip_var].isel({time_name: 0})
one_day_df = one_day.to_dataframe(name='precip_mm').reset_index()
print('One-day grid table shape:', one_day_df.shape)
display(one_day_df.head(10))

One-day grid table shape: (327825, 4)


,latitude,longitude,time,precip_mm
0,25.049861,-24.95014,1950-01-01,NaN
1,25.049861,-24.85014,1950-01-01,NaN
2,25.049861,-24.75014,1950-01-01,NaN
3,25.049861,-24.65014,1950-01-01,NaN
4,25.049861,-24.55014,1950-01-01,NaN
5,25.049861,-24.45014,1950-01-01,NaN
6,25.049861,-24.35014,1950-01-01,NaN
7,25.049861,-24.25014,1950-01-01,NaN
8,25.049861,-24.15014,1950-01-01,NaN
9,25.049861,-24.05014,1950-01-01,NaN


In [8]:
# Table-like view 2: one nearest point across time.
# This is a daily time series table, similar to a standard pandas dataset.
point_series = ds[precip_var].sel({lat_name: target_lat, lon_name: target_lon}, method='nearest')
point_df = point_series.to_dataframe(name='precip_mm').reset_index()
print('Nearest-point time-series shape:', point_df.shape)
display(point_df.head(10))

# Optional extra: very small subset by space and first 5 days.
small_subset = ds[precip_var].sel({
    lat_name: slice(bbox['min_lat'], bbox['max_lat']),
    lon_name: slice(bbox['min_lon'], bbox['max_lon']),
}).isel({time_name: slice(0, 5)})
small_subset_df = small_subset.to_dataframe(name='precip_mm').reset_index()
print('Small subset table shape:', small_subset_df.shape)
display(small_subset_df.head(10))

Nearest-point time-series shape: (27759, 4)


,time,longitude,latitude,precip_mm
0,1950-01-01,8.74986,53.049859,6.0
1,1950-01-02,8.74986,53.049859,6.8
2,1950-01-03,8.74986,53.049859,1.5
3,1950-01-04,8.74986,53.049859,0.6
4,1950-01-05,8.74986,53.049859,13.6
5,1950-01-06,8.74986,53.049859,1.0
6,1950-01-07,8.74986,53.049859,0.0
7,1950-01-08,8.74986,53.049859,0.0
8,1950-01-09,8.74986,53.049859,0.0
9,1950-01-10,8.74986,53.049859,0.0


Small subset table shape: (100, 4)


,time,latitude,longitude,precip_mm
0,1950-01-01,52.949859,8.54986,4.6
1,1950-01-01,52.949859,8.64986,4.3
2,1950-01-01,52.949859,8.74986,4.9
3,1950-01-01,52.949859,8.84986,5.1
4,1950-01-01,52.949859,8.94986,5.1
5,1950-01-01,53.049859,8.54986,5.7
6,1950-01-01,53.049859,8.64986,5.4
7,1950-01-01,53.049859,8.74986,6.0
8,1950-01-01,53.049859,8.84986,5.6
9,1950-01-01,53.049859,8.94986,5.4


In [9]:
# Extract two local series:
# 1) nearest grid cell to the target point,
# 2) all cells inside the selected bounding box.
da = ds[precip_var]

nearest = da.sel({lat_name: target_lat, lon_name: target_lon}, method="nearest")
region = da.sel({
    lat_name: slice(bbox["min_lat"], bbox["max_lat"]),
    lon_name: slice(bbox["min_lon"], bbox["max_lon"]),
})

nearest_lat = float(nearest[lat_name].values)
nearest_lon = float(nearest[lon_name].values)
# Count how many grid cells are available in the regional selection.
regional_cell_count = int(region.isel({time_name: 0}).count().values)

print(f"Nearest selected grid point: lat={nearest_lat:.4f}, lon={nearest_lon:.4f}")
print("Regional grid-cell count:", regional_cell_count)


Nearest selected grid point: lat=53.0499, lon=8.7499
Regional grid-cell count: 20


In [10]:
# Convert extracted xarray objects to pandas tables.
# nearest_df has one value per day from nearest grid cell.
# regional_df has one value per day from mean of all cells in the box.
nearest_df = nearest.to_dataframe(name="nearest_grid_precip_mm").reset_index()
regional_mean = region.mean(dim=(lat_name, lon_name))
regional_df = regional_mean.to_dataframe(name="regional_mean_precip_mm").reset_index()

df = nearest_df[[time_name, "nearest_grid_precip_mm"]].merge(
    regional_df[[time_name, "regional_mean_precip_mm"]],
    on=time_name,
    how="outer",
)

df = df.rename(columns={time_name: "date"})
df["date"] = pd.to_datetime(df["date"])

# For city-scale analysis we use regional mean as main series.
# This often gives a more stable local estimate than one grid cell alone.
df["selected_precip_mm"] = df["regional_mean_precip_mm"]
df = df.sort_values("date").reset_index(drop=True)

# Save the local extracted dataset to CSV.
output_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_csv, index=False)
print("Saved CSV:", output_csv)


Saved CSV: C:\Users\Amanda\Desktop\ISU\6. Semester\Virtual Reality and Optimization\Project\data\processed\bremen_daily_precipitation.csv


In [11]:
# Initial pandas checks.
# These are the first checks you should always run on a new extracted dataset.
print("Shape:", df.shape)
print("\nHead:")
display(df.head())

print("\nMissing values per column:")
print(df.isna().sum())

print("\nDescriptive stats:")
display(df[["nearest_grid_precip_mm", "regional_mean_precip_mm", "selected_precip_mm"]].describe())


Shape: (27759, 4)

Head:


,date,nearest_grid_precip_mm,regional_mean_precip_mm,selected_precip_mm
0,1950-01-01,6.0,5.735,5.735
1,1950-01-02,6.8,6.910,6.910
2,1950-01-03,1.5,1.775,1.775
3,1950-01-04,0.6,1.225,1.225
4,1950-01-05,13.6,13.730,13.730



Missing values per column:
date                       0
nearest_grid_precip_mm     0
regional_mean_precip_mm    0
selected_precip_mm         0
dtype: int64

Descriptive stats:


,nearest_grid_precip_mm,regional_mean_precip_mm,selected_precip_mm
count,27759.000000,27759.000000,27759.000000
mean,1.914446,1.973691,1.973691
std,3.883722,3.724568,3.724568
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,2.300000,2.560000,2.560000
max,68.500000,55.235004,55.235004


In [12]:
# Additional quality checks.
# Check whether dates are duplicated, missing, or if negative precipitation exists.
# Negative precipitation is usually not physically meaningful and should be investigated.
duplicate_dates = int(df.duplicated(subset=["date"]).sum())
expected_dates = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
missing_dates = expected_dates.difference(df["date"])
negative_count = int((df["selected_precip_mm"] < 0).sum())

print("Duplicate dates:", duplicate_dates)
print("Missing dates count:", len(missing_dates))
print("Negative selected precipitation count:", negative_count)
if len(missing_dates) > 0:
    print("First missing dates examples:", list(missing_dates[:10]))


Duplicate dates: 0
Missing dates count: 0
Negative selected precipitation count: 0


In [13]:
# Close the dataset to release resources.
ds.close()
print("Dataset closed.")


Dataset closed.
